# Slug Reclassification

Recomputes slug penetration using clean country denominators (excluding dissolved,
micro, failed, exclusion countries). Classifies slugs by UN region/subregion vectors.
Filters pool for re-clustering.

**Depends on:** `p01_01_country_missingness.ipynb` (must be run first, flags saved)

**Phase 1: Model Definition**

In [ ]:
using Revise
using InteractiveUtils

includet("phase1/functions/load_phase1.jl")

In [ ]:
using CSV, DataFrames

df = load_augmented_or_build()
meta_df = CSV.read("data/qog_metadata_plus2.csv", DataFrame)
println("Loaded: $(nrow(df)) rows, $(nrow(meta_df)) slugs")

## Step 1: Load Missingness Results

Re-run country missingness to get the status classifications.

In [ ]:
miss_result = run_country_missingness(df, meta_df; verbose=false)
println("Country statuses loaded: $(nrow(miss_result.status)) country-years")
sort(combine(groupby(miss_result.status, :country_status), nrow => :count), :count, rev=true)

## Step 2: Run Reclassification Pipeline

In [ ]:
reclass = run_slug_reclassification(df, meta_df, miss_result.status)

## Slug Disposition Summary

How many slugs in each category?

In [ ]:
# Dropped vs kept
dropped = filter(r -> !ismissing(r.drop_reason), reclass.penetration)
kept = filter(r -> ismissing(r.drop_reason), reclass.penetration)
println("Dropped: $(nrow(dropped))")
println(sort(combine(groupby(dropped, :drop_reason), nrow => :count), :count, rev=true))
println("\nKept: $(nrow(kept))")
println(sort(combine(groupby(kept, :temporal_profile), nrow => :count), :count, rev=true))

## UN Geographic Classification

In [ ]:
sort(combine(groupby(reclass.vectors, :un_geo_classification), nrow => :count), :count, rev=true)

## Global Slugs (Revised)

Slugs crossing the 95% population-weighted penetration threshold with the clean denominator.

In [ ]:
globals = filter(r -> r.un_geo_classification == "global", reclass.vectors)
println("Revised global slugs: $(nrow(globals))")
# Show by prefix
joined = leftjoin(globals[:, [:slug]], meta_df[:, [:slug, :prefix]], on=:slug)
prefix_counts = sort(combine(groupby(joined, :prefix), nrow => :count), :count, rev=true)
println("\nBy prefix:")
for r in eachrow(prefix_counts)
    println("  $(rpad(r.prefix, 15)) $(r.count)")
end

## Regional & Subregional Slugs

Slugs with geographically concentrated coverage.

In [ ]:
for geo_type in ["regional", "subregional"]
    subset = filter(r -> r.un_geo_classification == geo_type, reclass.vectors)
    println("\n=== $(uppercase(geo_type)) ($(nrow(subset)) slugs) ===")
    joined = leftjoin(subset[:, [:slug]], meta_df[:, [:slug, :prefix, :label]], on=:slug)
    prefix_counts = sort(combine(groupby(joined, :prefix), nrow => :count), :count, rev=true)
    for r in eachrow(first(prefix_counts, 10))
        println("  $(rpad(r.prefix, 15)) $(r.count)")
    end
end

## Sparse Slugs

In [ ]:
sparse = filter(r -> r.un_geo_classification == "sparse", reclass.vectors)
println("Sparse slugs: $(nrow(sparse))")
if nrow(sparse) > 0
    joined = leftjoin(sparse[:, [:slug]], meta_df[:, [:slug, :prefix, :label]], on=:slug)
    for r in eachrow(first(joined, 20))
        println("  $(rpad(r.slug, 25)) $(r.prefix)  $(r.label)")
    end
end

## Clustering Pool

Slugs remaining after removing global, regional, subregional, and sparse.

In [ ]:
pool = reclass.clustering_pool
println("Clustering pool: $(length(pool.clustering_pool)) slugs")
println("\nRemoved:")
println(pool.summary)

In [ ]:
# Preview pool slugs by prefix
pool_df = DataFrame(slug = pool.clustering_pool)
joined = leftjoin(pool_df, meta_df[:, [:slug, :prefix, :temporal_profile]], on=:slug)
prefix_counts = sort(combine(groupby(joined, :prefix), nrow => :count), :count, rev=true)
println("Pool slugs by prefix (top 20):")
for r in eachrow(first(prefix_counts, 20))
    println("  $(rpad(r.prefix, 15)) $(r.count)")
end

## Next: Re-Cluster

Run `p01_03_slug_clustering.ipynb` with the filtered pool from above.

In [ ]:
# Save for next notebook
# CSV.write("data/slug_reclassification.csv", reclass.penetration)
# CSV.write("data/clustering_pool.csv", DataFrame(slug=pool.clustering_pool))
# println("\u2705 Saved")